# Reading Data from Bronze

In [0]:
from pyspark.sql.functions import *

In [0]:
df_crm_cust = spark.table("workspace.bronze.crm_cust_info")
display(df_crm_cust.limit(5))

# Transforming Data

In [0]:
display(df_crm_cust)

## Transforming CRM cust table
- trim string values - good practice 
- normalization for status & gender
- column names are not friendly 

In [0]:
rename_map = {
    "cst_id" : "customer_id",
    'cst_key' : 'customer_key',
    'cst_firstname' : 'first_name',
    'cst_lastname' : 'last_name',
    'cst_marital_status' : 'marital_status',
    'cst_gndr' : 'gender',
    'cst_create_date' : 'creation_date'

}

In [0]:
df_crm_cust.columns

In [0]:
#checking -- not storing directly in df 

test = df_crm_cust.withColumn('cst_firstname', trim(col('cst_firstname')))

In [0]:
display(test.select('cst_firstname'))

In [0]:
# writing a loop to update all the fields in the table crm_cust to be trimmed and normalized 

for field in df_crm_cust.schema.fields:
  if field.dataType == StringType():
    df_crm_cust = df_crm_cust.withColumn(field.name, trim(col(field.name)))
display(df_crm_cust)

In [0]:
#Normalization
# First lets get all the unique values from married status & gender field

df_crm_cust.select('cst_marital_status').distinct().show()
df_crm_cust.select('cst_gndr').distinct().show()


In [0]:
#Lets get the total count of null values in the table

df_crm_cust.select([count(when(col(c).isNull(), c)).alias(c) for c in df_crm_cust.columns]).show()

In [0]:
#changing the married status to married or single & similarly for gender

df_crm_cust = (
    df_crm_cust
    .withColumn(
        "cst_marital_status",
        when(upper(col("cst_marital_status")) == 'S', "Single")
        .when(upper(col("cst_marital_status")) == 'M', "Married")
        .otherwise("n/a")

    )
    .withColumn(
        "cst_gndr",
        when(upper(col("cst_gndr")) == 'F', "Female")
        .when(upper(col("cst_gndr")) == 'M', "Male")
        .otherwise("n/a")
    )
)

In [0]:
display(df_crm_cust.limit(5))

In [0]:
#changing column names 

for old,new in rename_map.items():
  df_crm_cust = df_crm_cust.withColumnRenamed(old,new)
display(df_crm_cust.limit(5))

# Writing data to Silver Layer

In [0]:
df_crm_cust.write.mode("overwrite").format('delta').saveAsTable('workspace.silver.crm_cust')


In [0]:
%sql
select * from workspace.silver.crm_cust limit 5

Below code is better for orchestration 

from delta.tables import DeltaTable

- Check if table exists

`if spark.catalog.tableExists("workspace.silver.erp_cust")`:

    `target_table = DeltaTable.forName(spark, "workspace.silver.erp_cust")`
    
    `target_table.alias("target").merge(
        df_erp_cust.alias("source"),
        "target.customer_id = source.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()`
`else:
    # First run - create table
    df_erp_cust.write.format("delta").saveAsTable("workspace.silver.erp_cust")`

OR 

-  explicit DDL
`df_erp_cust.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("workspace.silver.erp_cust")`

In [0]:
# Return row count or status
row_count = spark.table("workspace.silver.erp_cust").count()
dbutils.notebook.exit(f"Success: {row_count} rows written")